# 5. Feature selection: multi-MSA alignment experiment <a id="5"></a>


## Table of contents

- [5.1 Setup & multi-seed run](#setup)
- [5.2 Comparison: runtime](#cmp-runtime)
- [5.3 Comparison: alignment coverage](#cmp-coverage)
- [5.4 Comparison: conservation overlap](#cmp-overlap)


## Backend map

How this notebook connects to `workflow/` modules:

![Backend map](images/backend_maps/08b-MultiMSAAlignmentExperiment.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
flowchart LR
  nb["08b-MultiMSAAlignmentExperiment"]
  m0["workflow.structural_alignment_experiment"]
  nb --> m0
  m1["workflow.utilities"]
  nb --> m1
```
-->


This notebook benchmarks two multiple-structure alignment methods — **FoldMason** and **MUSTANG** — on random subsamples of kinase structures from `Results/activation_segments/misaligned_filter/`, together with the BRAF reference `6UAN_chainD.pdb`.

Each method is run for **three random seeds** (`42`, `43`, `44`), each drawing **50** structures (+ reference). Comparison plots report **mean ± SEM** across those repeats.

| Experiment | Engine | Representation | Primary output |
|---|---|---|---|
| 1 — FoldMason | `foldmason easy-msa` | 3Di sequence + structure graph MSA | `msa_3di.fa` |
| 2 — MUSTANG | `mustang-3.2.4` | RMSD-based progressive MSA | `.afasta` |

Production full-dataset FoldMason conservation remains in `08a` / `08c`.


To get started, let's load some packages!


In [ ]:
from IPython.display import display, HTML

from workflow.structural_alignment_experiment import (
    run_all_seeds,
    plot_runtime_comparison,
    plot_coverage_comparison,
    plot_conservation_overlap,
)
from workflow.utilities import braf_res


## 5.1 Setup & multi-seed run <a id="setup"></a>

We draw **50 structures at random** (without replacement) from `Results/activation_segments/misaligned_filter/` and add the BRAF reference once. The same sample is fed to FoldMason and MUSTANG within each seed. Seeds `42`, `43`, and `44` provide three independent repeats for mean ± SEM summaries.


In [ ]:
SEEDS = (42, 43, 44)
N_SAMPLE = 50

RECONSTR_DIR = "Results/activation_segments/misaligned_filter/"
REFERENCE_PDB = "6UAN_chainD.pdb"
EXPERIMENT_DIR = "Results/Experiments/structural_alignment"
MUSTANG_BIN = "/home/marmatt/Downloads/MUSTANG_v3.2.4/bin/mustang-3.2.4"

per_seed_df = run_all_seeds(
    seeds=SEEDS,
    reconstr_dir=RECONSTR_DIR,
    reference_pdb=REFERENCE_PDB,
    experiment_dir=EXPERIMENT_DIR,
    n_sample=N_SAMPLE,
    mustang_bin=MUSTANG_BIN,
    conservation_threshold=0.70,
    show_conservation_plots=False,
)

display(per_seed_df)


## 5.2 Comparison: runtime <a id="cmp-runtime"></a>

Wall-clock runtime for FoldMason vs MUSTANG, aggregated as **mean ± SEM** over the three seeds.


In [ ]:
runtime_summary = plot_runtime_comparison(
    per_seed_df,
    experiment_dir=EXPERIMENT_DIR,
    show=True,
)
display(runtime_summary)


## 5.3 Comparison: alignment coverage <a id="cmp-coverage"></a>

Summary coverage metrics — alignment length, fully aligned columns, and mean gap fraction — as grouped bars with **mean ± SEM** (no per-structure gap-fraction histograms).


In [ ]:
coverage_summary = plot_coverage_comparison(
    per_seed_df,
    experiment_dir=EXPERIMENT_DIR,
    show=True,
)
display(coverage_summary)


## 5.4 Comparison: conservation overlap <a id="cmp-overlap"></a>

Overlap of residues conserved at ≥70 % in each method (FoldMason only / both / MUSTANG only), with **mean ± SEM** counts and mean Jaccard ± SEM.


In [ ]:
overlap_summary = plot_conservation_overlap(
    per_seed_df,
    experiment_dir=EXPERIMENT_DIR,
    show=True,
)
display(overlap_summary)
